# Validación del conteo contra Google Maps / ESRI Wayback

**Metodología**: se muestrean N polígonos al azar. Para cada uno se generan URLs a Google Maps (satélite 2024) y a ESRI Wayback (histórico). El usuario cuenta edificios manualmente en cada muestra y los compara con la estimación del sistema.

Criterio de aceptación: si el 80% de las muestras cae dentro de la banda ±15%, el conteo es aceptable para Fase 1.

Fecha: 2026-04-22

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# Función para muestrear polígonos y generar URLs de inspección manual
def muestrear_polignos(
    ruta_geojson: Path = Path('../config/poligonos.geojson'),
    n: int = 5,
    seed: int = 42,
) -> pd.DataFrame:
    """Devuelve un DataFrame con N polígonos y URLs a Google Maps / ESRI Wayback."""
    gdf = gpd.read_file(ruta_geojson)
    muestra = gdf.sample(n=min(n, len(gdf)), random_state=seed).copy()
    centroides = muestra.geometry.centroid
    muestra['lon'] = centroides.x
    muestra['lat'] = centroides.y

    # Google Maps: vista satélite con marker
    muestra['url_google_maps'] = muestra.apply(
        lambda r: f'https://www.google.com/maps/@{r.lat:.5f},{r.lon:.5f},17z/data=!3m1!1e3',
        axis=1,
    )
    # ESRI Wayback: histórico de imágenes satelitales
    muestra['url_esri_wayback'] = muestra.apply(
        lambda r: (
            f'https://livingatlas.arcgis.com/wayback/#active=44116'
            f'&mapCenter={r.lon:.5f}%2C{r.lat:.5f}%2C17'
        ),
        axis=1,
    )
    return muestra[['id', 'nombre', 'categoria', 'lat', 'lon',
                    'url_google_maps', 'url_esri_wayback']]

muestras = muestrear_polignos(n=5)
for _, fila in muestras.iterrows():
    print(f"\n[{fila['id']}] {fila['nombre']} ({fila['categoria']})")
    print(f'  Google Maps: {fila.url_google_maps}')
    print(f'  ESRI Wayback: {fila.url_esri_wayback}')

In [ ]:
# Ground truth manual: completar con el conteo hecho manualmente en Google Maps (año 2024)
# Clave: poligono_id, valor: n_edificios_manual
GROUND_TRUTH_MANUAL = {
    # 'itaembe_mini': 2100,    # estimación del usuario
    # 'itaembe_guazu': 1450,
    # 'chacra_32': 800,
    # 'villa_cabello': 1900,
    # 'el_brete': 120,
}

# Leer estimación del sistema (serie temporal 2024-07)
RUTA_SERIE = Path('../data/processed/conteos/serie_temporal.csv')
if RUTA_SERIE.exists():
    serie = pd.read_csv(RUTA_SERIE)
    fecha_comparacion = '2024-07'
    sistema = serie[serie['fecha'] == fecha_comparacion].set_index('poligono_id')
    print(f'Estimaciones del sistema a fecha {fecha_comparacion}:')
    print(sistema[['n_edificios_min', 'n_edificios_estimado', 'n_edificios_max']])
else:
    sistema = pd.DataFrame()
    print('serie_temporal.csv no existe - correr 20_contar_techos.py primero.')

In [ ]:
# Comparación tabular y gráfico de residuos
if len(GROUND_TRUTH_MANUAL) and len(sistema):
    comparacion = []
    for pol_id, n_manual in GROUND_TRUTH_MANUAL.items():
        if pol_id not in sistema.index:
            continue
        fila = sistema.loc[pol_id]
        n_est = int(fila['n_edificios_estimado'])
        comparacion.append({
            'poligono_id': pol_id,
            'n_manual': int(n_manual),
            'n_estimado': n_est,
            'n_min': int(fila['n_edificios_min']),
            'n_max': int(fila['n_edificios_max']),
            'error_pct': 100.0 * (n_est - n_manual) / n_manual,
            'dentro_banda_15pct': abs(n_est - n_manual) / n_manual <= 0.15,
        })
    comp_df = pd.DataFrame(comparacion)
    print(comp_df.to_string(index=False))
    print()
    pct_ok = 100.0 * comp_df['dentro_banda_15pct'].mean()
    print(f'Muestras dentro de banda ±15%: {pct_ok:.1f}%')

    # Gráfico de residuos: observado vs esperado con banda ±15%
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(
        comp_df['n_manual'],
        comp_df['n_estimado'],
        s=80,
        color='#1a3a5c',
        edgecolors='white',
        zorder=3,
    )
    for _, r in comp_df.iterrows():
        ax.annotate(
            r['poligono_id'],
            (r['n_manual'], r['n_estimado']),
            fontsize=8,
            xytext=(5, 5),
            textcoords='offset points',
        )
    lim_max = max(comp_df['n_manual'].max(), comp_df['n_estimado'].max()) * 1.1
    eje = np.linspace(0, lim_max, 100)
    ax.plot(eje, eje, '--', color='grey', label='Identidad y=x', zorder=1)
    ax.fill_between(
        eje, eje * 0.85, eje * 1.15, alpha=0.2, color='#c97d3c',
        label='Banda ±15%', zorder=0,
    )
    ax.set_xlabel('Conteo manual (ground truth)')
    ax.set_ylabel('Conteo del sistema')
    ax.set_title('Observado vs. esperado - banda ±15%')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print(
        'Completá GROUND_TRUTH_MANUAL arriba con tus conteos manuales y '
        'asegurate de tener serie_temporal.csv.'
    )

## Criterio de aceptación

Si el **80% o más de las muestras cae dentro de la banda ±15%**, el conteo se considera aceptable para Fase 1.

Si esto no se cumple:

1. Revisar polígonos que caen fuera de banda - ¿son zonas con vegetación densa, sombras de árboles, o nubes?
2. Ajustar `ndbi_threshold` (probar 0.05-0.10 si hay muchos falsos positivos) o `ndvi_threshold` (bajar a 0.25 si la zona es muy verde).
3. Si el problema es dataset-wide, considerar subir `confidence_min` a 0.8.
4. Documentar en `METODOLOGIA.md` la desviación observada y qué ajuste se aplicó.

En Fase 2 (con Planet NICFI 4.7m), se espera reducir esta banda a ±10%.